# PriceWise Pricing Recommendation Research Workbook

## Methodology summary

PriceWise has two models and one pricing engine.

```Historical sales → 1. ASP benchmark → Reference price → Candidate prices```

**Features:** ASP7, ASP30 [1]



```Candidate prices + weekly demand inputs → 2. Demand model → Expected demand at each price```

**Features:**
**Run rate at R** [8] (sales history [2], product history [3], buyer and channel patterns [4], competitor context at R [5])  
× **Weekly adjustment** (market-weighted month, quarter and event [9], scheduled promotion [6], seasonality × sales momentum [7])  
× **Candidate-price adjustment** (candidate price relative to R [6], price change × promotion in the US [7], elasticity parameter [10]).



``Expected demand + inventory + cost + pricing rules → 3. Pricing engine → Expected units sold and recommended price``

**Features:** Stock and clearance [11], Competitor-band rule [5], Margin [12], Unit target and price bounds [13], Inventory interactions [14]



``Recommendation + candidate comparison → Dashboard``










##### Notes on the inputs
1. **ASP7 and ASP30.** Average transaction prices over the previous 7 and 30 days, weighted by units sold. Their blend gives the reference price R. For SKUs with sparse price history, Jade’s hierarchical price model can be an alternative way to estimate the reference price R. The current model uses the family-weighted ASP benchmark. [Reference Price Report.docx](Reference%20Price%20Report.docx)
2. **Sales history.** Recent volume, sales pace, and selling frequency help estimate normal weekly sales.
3. **Calendar and product history.** Fiscal dates, holidays, prior-year family sales, and time since the SKU first sold help put the upcoming week in context.
4. **Buyer and channel patterns.** Sales shares show whether the SKU sells broadly or depends on a few buyers, channels, or regions.
5. **Competitor inputs.** Comparable offers tell us how our candidate price compares with alternatives. The competitor band can also inform an enabled pricing rule.
6. **Price and promotion context.** Compare the candidate with ASP, the current offer, and list price. Use scheduled promotions and recent offer changes to give those gaps context.
7. **Demand interactions.** Test combined effects, such as whether a competitor premium matters more during a promotion or whether a price move matters more for a price-sensitive SKU.
8. **Run rate at R.** Expected units in an ordinary week if we charge the ASP reference price R. A run rate of 100 means we start from 100 expected units before adjusting for this week's calendar or candidate price.
9. **Calendar effects.** Monthly seasonality adjusts demand for an ordinary week in that month. Event lift captures the extra change for a specific week, such as Black Friday, relative to an ordinary week in the same month. Both can change demand at the same price.
10. **Price elasticity.** The percentage change in demand associated with a small change in price, holding other inputs fixed. An elasticity of −2 means a 1% price increase corresponds to about 2% fewer units. The report’s hierarchical elasticity method estimates ε at SKU level when enough history is available, then falls back to product family and then to segment × product type. The demand model uses ε together with the reference price R. [Pricewise_Update_Elasticity_Seasonality_Event.docx](Pricewise_Update_Elasticity_Seasonality_Event.docx)
11. **Stock and clearance.** Sellable stock caps expected units sold. Stock cover, age, and inbound units help assess clearance pressure. Leftover stock is calculated after estimating sales at each candidate price.
12. **Margin.** Candidate price minus unit cost gives gross profit per unit. Gross margin share expresses it as a share of price.
13. **Unit target and price bounds.** The target gives the engine a sales goal. Floors and ceilings define the allowed price range.
14. **Inventory interactions.** Combine stock pressure with slowing sales or an offer above competitors to describe why clearance might be more urgent.


### Decision and scope
 Each Monday, compare candidate prices with holding the current offer. The initial scope covers the US and Canada and the five groups observed in the exploratory analysis: Enterprise Notebook, Desktop, and Workstations, and Personal Notebook and Desktop. The inital EDA found differences in *event lift* and *price sensitivity* across these groups, so the plan tracks weekly demand by country, SKU, and segment. Where segments share an offer or stock pool, evaluate the common price and apply the stock limit across those segments. 

### Research questions

1. At the ASP reference price, how much do recent sales velocity, sales frequency, monthly seasonality, and events such as Black Friday improve forecasts of weekly SKU sales?
   - **Demand model:** Identify which features improve the weekly unit forecast at the ASP reference price.
2. How do expected weekly units change when we quote 5% above or below the ASP reference, and how does that response vary by product family, customer segment, or price relative to comparable competitors?
   - **Demand model:** Use the estimated price response to turn the reference-price forecast into expected units at each candidate price.
3. Compared with holding the current offer, which candidate price gives the highest expected gross profit within the approved price limits, and how does the choice change under different inventory levels or clearance targets?
   - **Pricing engine → Dashboard:** Select a feasible price and show how its expected units and gross profit compare with holding the current offer.

### Technical Details and Economic Principles

#### 1. ASP Benchmark

$$
R=w\,ASP_7+(1-w)\,ASP_{30}
$$
- **ASP7** → Responds faster when prices move.
- **ASP30** → Draws on more sales and is less sensitive to a single unusual week.



[Reference_Price_Exploration.ipynb](Reference_Price_Exploration.ipynb)

#### 2. Demand Model

$$
\begin{aligned}
Q(p)
&= \underbrace{B_t}_{\text{Run rate at }R}
   \times \underbrace{A_t}_{\text{This week's adjustment}}
   \times \underbrace{P_t(p)}_{\text{Candidate-price adjustment}} \\[6pt]
&= B_t \times
\underbrace{\exp(w_m m_t+w_q q_t+w_c c_t+w_o o_t)}_{\text{Calendar and campaign (seasonality)}}
\\
&\quad \times
\underbrace{\exp(w_b b_t(p)+w_h h_t(p))}_{\text{price relative to R, competitor price, elasiciticy, and promotion interaction}}.
\end{aligned}
$$

- **$B_t$** → Ordinary-week demand at the reference price $R$.
- **$A_t$** → Same price, different week. Adjusts for seasonality, events, and promotions.
- **$P_t(p)$** → Same week, different price. Elasticity determines how strongly demand responds. When $p=R$, $P_t(R)=1$.

Here m, q, c and o are the fitted month, quarter, campaign and remaining weekly log contributions. The last group includes time of year, events and seasonal momentum. The two price contributions are base price response and the extra response during promotions; both equal zero at R.

| Market | Month | Quarter | Campaign | Other weekly effects | Base price | Promo × price |
|---|---:|---:|---:|---:|---:|---:|
| US | 1 | 0.5 | 0.5 | 1 | 1 | 1 |
| CA | 0.5 | 0.5 | 1 | 1 | 1 | 0 |

These multipliers scale fitted log contributions. A weight of 0.5 takes the square root of the corresponding demand multiplier; 0 makes it neutral. Seven earlier quarters selected the weights by MAE. [model_adjustment.ipynb](model_adjustment.ipynb)

[Pricewise_Update_Elasticity_Seasonality_Event.docx](Pricewise_Update_Elasticity_Seasonality_Event.docx)

[demand_model.ipynb](demand_model.ipynb)

#### 3. Pricing Engine

**Economic principle.** A higher price increases gross profit per unit but can reduce units sold. With constant unit cost $c_t$, smooth downward-sloping demand, and an unconstrained interior optimum:

$$
\frac{d}{dp}\big[(p-c_t)Q(p)\big]
=
\underbrace{Q(p)}_{\text{Margin gained on units still sold}}
+
\underbrace{(p-c_t)Q'(p)}_{\text{Margin lost on units no longer sold}}
=
0
\quad\Longleftrightarrow\quad MR=MC.
$$

**How PriceWise selects a price.** Inventory and pricing rules can put the best price at a boundary. The engine therefore evaluates each feasible candidate directly:

$$
\begin{aligned}
\underbrace{\widehat U_t(p)}_{\text{Expected units sold}}
&=
\min\!\left\{
\underbrace{Q(p)}_{\text{Demand Model output}},
\underbrace{I_t}_{\text{Sellable inventory}}
\right\},\\[6pt]
\underbrace{\widehat{\mathrm{GP}}_t(p)}_{\text{Expected gross profit}}
&=
\underbrace{(p-c_t)}_{\text{Gross profit per unit}}
\times
\underbrace{\widehat U_t(p)}_{\text{Expected units sold}},\\[6pt]
\underbrace{p_t^*}_{\text{Recommended price}}
&\in
\underset{p\in\mathcal P_t^{\mathrm{feasible}}}{\operatorname{arg\,max}}
\widehat{\mathrm{GP}}_t(p).
\end{aligned}
$$

- **$Q(p)$** → Expected demand at candidate price $p$ in week $t$.
- **$I_t$** → Sellable inventory. Caps how many units can sell.
- **$c_t$** → Unit cost. Converts candidate price into gross profit per unit.
- **$\mathcal P_t^{\mathrm{feasible}}$** → Candidates that meet the applicable price rules, such as a floor, ceiling, or mandatory competitor band. If the unit target is a hard requirement, candidates must also satisfy $\widehat U_t(p)\ge T_t$, where $T_t$ is the target.
- **$p_t^*$ and $\widehat U_t(p_t^*)$** → The recommended price and expected units sold at that price. The dashboard compares both with holding the current offer. If the feasible set is empty, the engine flags the decision for review.



[price_engine.ipynb](price_engine.ipynb)


### Steps at a glance

| Step | Progress |
|---|:---:|
| 1. Define the pricing decision | ✓ |
| 2. Set the initial hypotheses | ✓ |
| 3. Construct features | ✓ |
| 4. Check Historical Data for B, A, and P | ✓ |
| 5. Fit B, A, and P and backtest demand | |
| 6. Select the demand model and compare candidate prices | |

## 2. Form hypotheses

**Sales history helps set the starting volume.** A recent pickup may carry into next week. The earlier EDA also found short SKU histories, so we will test longer windows or family pooling for thin products.

**Calendar effects create different demand conditions at the same price.** Both markets showed November volume increases. The team's event analysis suggests that named holiday weeks explain part of the peak. This motivates separate month and event terms.

**Buyers may respond to price and comparable offers.** Lower realized prices coincided with higher volume in prior EDA. A proposed price above comparable alternatives may also affect demand. The first relationship is observational, and actual competitor effects require matched offer data. Both can be tested as assumptions in the demo.

**Inventory and economics change what is worth recommending.** Stock limits how much can sell, and excess stock can make clearance more valuable. Unit cost and approved price limits affect which choices are feasible or profitable. These inputs matter even if they do not improve demand prediction.


## 3. Construct features

Build a weekly panel from eligible invoices and reconcile units and revenue to the source. Verify the currency and when each field becomes available. Use only inputs available at the Monday cutoff. Weeks with no invoice remain unobserved unless listing, stock availability, and source coverage support recording zero units.

### 3.1 ASP benchmark

 $R=w\,ASP_7+(1-w)\,ASP_{30}$, using the windows before Monday. The weight is fitted by country, product type, segment, and family.

[Reference_Price_Exploration.ipynb](Reference_Price_Exploration.ipynb) · [Reference Price Report.docx](Reference%20Price%20Report.docx)

**Current reference-price rules**

- ASP7, ASP30, and a fitted group weight are available → Blend the SKU’s own two ASPs using the shared group weight.
- ASP7 or the group weight is missing, and ASP30 is available → Use the SKU’s ASP30. Even one eligible sale can produce an ASP30, so a fallback price may still have thin support.
- ASP30 is missing → The current benchmark supplies no reference price. Jade’s hierarchical price model can be tested as a fallback.

**TODO · what if we don't have ASP30 or ASP7 for certain SKU?**

- [ ] Obtain Jade’s model or its timestamped reference-price predictions.
- [ ] Compare Jade’s predictions with ASP30 on the same held-out sparse SKU-weeks where both can provide a price. Use the same currency, cutoff, and realized weekly ASP.
- [ ] For SKU-weeks with no ASP30, report how many additional prices Jade’s model can provide and check their errors where later sales are observed. Verify whether those SKU-weeks also have a usable demand baseline before counting them as additional recommendation coverage.


### 3.2 Candidate features

The list starts from the hypotheses in Step 2. Within each signal we consider levels, windows, relative measures, and selected interactions. The formulas are definitions to implement and screen. They are hypotheses until evaluated.

**Notation.** $t$ is the decision cutoff; $u_d$ is recorded units $d$ days back; $U_d=\sum_{i=1}^d u_i$ and $v_d=U_d/d$. $Q_j$ is units in the $j$th prior completed week, and $N_{28}$ is recorded invoice lines. $s_{g,d}$ is the unit share of group $g$ and $F$ is the same market, segment, and product family. $k$ is week of year and $0<\lambda<1$. $B$ is normal weekly units, $R$ is reference ASP, $P_0$ is current offer, $p$ is the candidate, $c$ is unit cost, $L$ is list price, and $T$ is the units target. $I$ is sellable inventory, $A$ is confirmed inbound inventory, $H$ is the evaluation horizon, and $S_t$ is the seasonal factor. Unless otherwise stated, trailing customer and regional shares use 28 days.


#### Recent sales

Compare time windows because recent changes may persist while longer histories smooth noisy weeks. Log volume compresses the long tail.

| # | Feature | Calculation |
|---|---|---|
| 1 | Units in last 7 days | $U_7$ |
| 2 | Units in last 14 days | $U_{14}$ |
| 3 | Units in last 28 days | $U_{28}$ |
| 4 | Units in last 56 days | $U_{56}$ |
| 5 | Log of 28-day units | $\log(1+U_{28})$ |
| 6 | Recent sales pace vs. typical pace | $v_7/v_{28}$ |
| 7 | Recent pace minus preceding pace | $v_7-(U_{28}-U_7)/21$ |
| 8 | Change from previous week | $Q_1-Q_2$ |
| 9 | Four-week sales trend | $\operatorname{slope}(Q_4,Q_3,Q_2,Q_1)$ |
| 10 | Smoothed weekly sales | $(Q_1+\lambda Q_2+\lambda^2Q_3+\lambda^3Q_4)/(1+\lambda+\lambda^2+\lambda^3)$ |


#### How regularly it sells

Two SKUs with the same total units may have different run rates if one sells steadily and the other depends on a few large orders.

| # | Feature | Calculation |
|---|---|---|
| 11 | Weeks with recorded sales | $\sum_{j=1}^{8}\mathbf1[Q_j>0]$ |
| 12 | Days since last recorded sale | $t-t_{\mathrm{last\ sale}}$ |
| 13 | Selling days in last 28 days | $\sum_{d=1}^{28}\mathbf1[u_d>0]$ |
| 14 | Invoice lines in last 28 days | $N_{28}$ |
| 15 | Units per invoice line | $U_{28}/N_{28}$ |
| 16 | Share of invoice lines with several units | $\#\{\ell:q_\ell>1\}/N_{28}$ |
| 17 | Week-to-week sales variability | $\operatorname{sd}(Q_1,\ldots,Q_8)$ |
| 18 | Variability relative to average sales | $\operatorname{sd}(Q_1,\ldots,Q_8)/\overline Q$ |
| 19 | Share sold on the busiest day | $\max_{d\le 28}u_d/U_{28}$ |
| 20 | Share of eight-week units sold in last two weeks | $U_{14}/U_{56}$ |


#### Time of year and product history

November spikes and short SKU histories motivate calendar and lifecycle measures. Family sales can supply context when a SKU has little history.

| # | Feature | Calculation |
|---|---|---|
| 21 | Fiscal month | $m_f(t)$ |
| 22 | Fiscal quarter | $q_f(t)$ |
| 23 | Position in year, sine | $\sin(2\pi(k-1)/52.18)$ |
| 24 | Position in year, cosine | $\cos(2\pi(k-1)/52.18)$ |
| 25 | Days until next relevant holiday | $t_{\mathrm{holiday}}-t$ |
| 26 | Event week flag | $\mathbf1[t\in \mathcal W_{\mathrm{event}}]$ |
| 27 | Family sales index in the same week last year | $Q_{F,k,\mathrm{prior\ year}}/\overline Q_{F,\mathrm{prior\ year}}$ |
| 28 | Recent change in family sales | $U_{F,28}/(U_{F,56}-U_{F,28})-1$ |
| 29 | Days since the SKU first recorded a sale | $t-t_{\mathrm{first\ sale}}$ |
| 30 | SKU share of its family's units | $U_{28}/U_{F,28}$ |


#### Who buys and where sales come from

A SKU may depend on one buyer, sales channel, or region. These are hypotheses about the stability of sales. Verify customer IDs before using them as buyer identifiers.

| # | Feature | Calculation |
|---|---|---|
| 31 | Share sold through Direct | $s_{\mathrm{Direct},28}$ |
| 32 | Share sold through Marketplace | $s_{\mathrm{Marketplace},28}$ |
| 33 | Dependence on a small number of channels | $\sum_c s_{c,28}^2$ |
| 34 | Number of customer IDs | $\#\{i:U_{i,28}>0\}$ |
| 35 | Share bought by the largest customer | $\max_i s_{i,28}$ |
| 36 | Share bought by returning customers | $\sum_{i:U_{i,56}-U_{i,28}>0}s_{i,28}$ |
| 37 | Dependence on a few states or provinces | $\sum_g s_{g,28}^2$ |
| 38 | Change in Direct sales share | $s_{\mathrm{Direct},28}-s_{\mathrm{Direct},29:56}$ |
| 39 | Sales rank within its family | $\operatorname{percentile}_F(U_{28})$ |
| 40 | SKU momentum relative to family | $v_7/v_{28}-v_{F,7}/v_{F,28}$ |


#### Stock and clearance

Stock can cap sales, while excess or aging stock creates a reason to consider clearance. Weeks of supply expresses stock relative to demand.

| # | Feature | Calculation |
|---|---|---|
| 41 | Units available to sell | $I$ |
| 42 | Log of available units | $\log(1+I)$ |
| 43 | Weeks of supply | $I/B$ |
| 44 | Weeks of supply above target | $I/B-W^*$ |
| 45 | Low stock flag | $\mathbf1[I/B<W_{min}]$ |
| 46 | Share of stock above age threshold | $I_{\mathrm{old}}/I$ |
| 47 | Median age of stock | $\operatorname{median}(\mathrm{age\ of\ sellable\ units})$ |
| 48 | Confirmed inbound stock in next 14 days | $\sum_{d=1}^{14}A_d$ |
| 49 | Days to confirmed replenishment | $t_{\mathrm{receipt}}-t$ |
| 50 | Projected leftover stock over the horizon | $\max(0,I+A_H-\widehat Q_H)$ |


#### Prices of comparable products

Buyers may substitute toward comparable products. Match quality and quote freshness affect how much weight the comparison deserves.

| # | Feature | Calculation |
|---|---|---|
| 51 | Number of comparable quotes | $\#\{p_j\}$ |
| 52 | Typical comparable price | $C=\operatorname{median}_j(p_j)$ |
| 53 | Current price vs. typical comparable price | $P_0/C-1$ |
| 54 | Current price vs. lowest comparable price | $P_0/\min_j p_j-1$ |
| 55 | Candidate price vs. typical comparable price | $p/C-1$ |
| 56 | Spread of comparable prices | $\operatorname{IQR}_j(p_j)/C$ |
| 57 | Above the competitor band | $\mathbf1[P_0>C_{\mathrm{upper}}]$ |
| 58 | Change in typical comparable price | $C/C_{\mathrm{prev7}}-1$ |
| 59 | Days since the last comparable quote | $t-t_{\mathrm{quote}}$ |
| 60 | Share of comparable offers in stock | $\#\{\mathrm{in\ stock\ offers}\}/\#\{p_j\}$ |


#### Price, profitability, and rules

Price-gap measures describe the proposed move. Margins, bounds, and targets turn it into a business decision. A target can affect the objective without changing demand.

| # | Feature | Calculation |
|---|---|---|
| 61 | Candidate vs. ASP reference | $p/R-1$ |
| 62 | Candidate vs. current offer | $p/P_0-1$ |
| 63 | Gross profit per unit | $p-c$ |
| 64 | Gross margin share | $(p-c)/p$ |
| 65 | Discount from list price | $1-p/L$ |
| 66 | Days since the offer last changed | $t-t_{\mathrm{last\ offer\ change}}$ |
| 67 | Promotion scheduled flag | $\mathbf1[\mathrm{promotion\ scheduled}]$ |
| 68 | Unit target relative to normal sales | $T/B$ |
| 69 | Distance above allowed floor | $p-P_{min}$ |
| 70 | Distance below allowed ceiling | $P_{max}-p$ |


#### Signals that may matter together

Interactions test specific hypotheses. Clearance pressure may matter more when sales are slowing, and a competitor premium may matter differently during a promotion.

| # | Feature | Calculation |
|---|---|---|
| 71 | Stock cover with slowing sales | $(I/B)\max(0,1-v_7/v_{28})$ |
| 72 | Excess stock while priced above competitors | $\max(0,I-BW^*)\mathbf1[P_0>C_{\mathrm{upper}}]$ |
| 73 | Candidate premium during promotion | $(p/C-1)\mathbf1[\mathrm{promotion\ scheduled}]$ |
| 74 | Seasonal demand with sales momentum | $S_t(v_7/v_{28})$ |
| 75 | Price move scaled by estimated sensitivity | $\widehat{\epsilon}_{\mathrm{prior}}\log(p/P_0)$ |


**Data and missing values.** Stock, stock age, inbound inventory, and comparable offer inputs may be synthesized for the dashboard. Missing current quotes, promotions, and targets may be supplied as labeled scenario assumptions. Costs and approved price limits require confirmation for business use. A zero or unknown denominator leaves the ratio missing. Windows overlapping incomplete coverage carry a flag. Sparse support for elasticity is recorded alongside the estimate.

### 3.3 Demo inputs where records are missing

After estimating a starting weekly run rate, generate seeded inventory and comparable offers for eligible real SKUs. Cover excess, balanced, and low stock, alongside below-market, in-band, and above-market offers. Include missing comparisons and a conflict between clearance and a hard floor. Save scenario ID, seed, cutoff, currency, and input provenance. Simulated outcomes are conditional on these assumptions.

**Weekly campaigns.** Preserve holiday dates. The synthetic back-to-school campaign covers US and Canadian Personal Notebooks on Mondays in August through September 7. Within an eligible window, assign 5%, 10% or 15% discounts when weekly sales reach 1.2, 1.5 or 2 times the prior 28-day weekly average. These are demo thresholds. For incomplete weeks use prior seven-day units. Store the reason and segment participation separately from the shared market–SKU quote. Use the deepest eligible segment discount once in that quote. The demand model reads participation for A and evaluates the quote through P. Because completed-week assignments depend on realized units, interpret their fit as retrospective. [demand_model.ipynb](demand_model.ipynb) · [campaign_windows.csv](synthesis%20data/campaign_windows.csv) · [weekly_campaign_participation.csv](synthesis%20data/weekly_campaign_participation.csv) · [weekly_offer_and_promotion_history.csv](synthesis%20data/weekly_offer_and_promotion_history.csv)

## 4. Check Historical Data for B, A, and P

Start with coverage before interpreting demand. The prior full-catalog scan covered 5.56 million lines. The team report found unusually few customers and far fewer transactions in October–December 2025. Recheck this in scope, keep the flagged records for audit, and compare fitting with and without the affected weeks where meaningful. A trailing window that crosses the gap needs a coverage flag.

**Run rate.** Compare simple trailing averages with a smoothed baseline. Look at coverage and sparsity first, then ask whether faster recent sales tend to carry into the next week. Keep a family fallback for thin SKUs and record which level produced the estimate.

**Seasonality and events.** Ask how a normal August week differs from a typical week, then ask how Black Friday differs from an ordinary week in the same month. The team estimates these effects by market and segment × product type. It reports about 1.4× August demand and about 3× Black Friday demand for Personal Notebooks. Holiday estimates rely on two usable years and are pending reproduction here. [Pricewise_Update_Elasticity_Seasonality_Event.docx](Pricewise_Update_Elasticity_Seasonality_Event.docx)

**Price and units.** Compare prices and volumes within comparable market-weeks. The team reports pooled elasticities of roughly −2.45 in the US and −1.63 in Canada. Its fallback moves from SKU to family to segment × product type. The written report motivates this design. The estimates and implementation still need replication. Same-week comparisons can absorb shared demand shocks, while SKU promotions, availability, and changing buyer mix can still bias the relationship. [Pricewise_Update_Elasticity_Seasonality_Event.docx](Pricewise_Update_Elasticity_Seasonality_Event.docx)

## 5. Fit B, A, and P and backtest demand

Join the reference price available each Monday to each eligible recorded-sales week. Fit the baseline, calendar and price terms jointly across historical quote levels. Use the market-specific contribution weights in Section 2 of Technical Details to predict at the original quote and each candidate.

Seven earlier quarters selected the adjustment strengths by MAE. On the June–August 2026 comparison the adjusted complete model scored 19.856 units in the US and 5.304 in Canada. The US complete model ranked first; Canada’s B × A scored 5.291. Completed-week campaigns use realized sales, so these results measure retrospective scenario fit. [model_adjustment.ipynb](model_adjustment.ipynb) · [demand_model.ipynb](demand_model.ipynb)

## 6. Select the demand model and compare candidate prices

Use the selected feature combination and fitted weights for each market. Export B, A, P and expected demand for each candidate, including the current offer. Keep physical stock limits, margin floors, and units targets in the decision rules. Use projected leftover stock and candidate margin as outputs of the candidate comparison. When related transformations perform similarly, prefer the representation a reviewer can interpret.

For a neutral run rate $B$, an illustrative demand calculation is

$$
\widehat Q(p)=B_t\,A_t\,P_t(p),
\qquad
\widehat{\mathrm{GP}}(p)=(p-c)\min\{\widehat Q(p),I\}.
$$

Sum segment demand before applying a shared stock cap, and account for segment-specific costs if applicable. Compare feasible candidates with holding the current price. When there is no feasible price or the result depends heavily on assumptions, flag the case for review.

**Dashboard.** Select a SKU and week, then show current offer, ASP, feasible range, competitor range, recommendation, and expected units and gross profit. Toggle inventory pressure, units targets, or competitor rules and recalculate. Physical stock and hard constraints stay active. Display the score curve, profit-only comparison, binding rule, and source of synthetic inputs. Test that each chart and recommendation matches the same candidate table.


For the dashboard campaign toggle, keep the same SKU, week, candidate prices, cost, inventory pool and clearance policy. Toggle segment campaign participation in A, evaluate P at each fixed candidate, and compare feasible gross profit with holding price. Report a price change only when the engine selects a different candidate.

## Results and report handoff

Add a compact result and interpretation beside each tested method as work progresses. Record the sample, dates, comparison, and result, then link the supporting table or figure. The final report can follow the same six-step sequence and summarize what held up, where assumptions were needed, and how the dashboard behaved.

**Next working session.** Compare candidate prices using the calibrated market demand curves and the applicable inventory, cost and price rules. [demand_model.ipynb](demand_model.ipynb)